# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [22]:
!pip install datasets huggingface_hub --quiet

from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [23]:
dataset = load_dataset("FlyRank/internship-warehouse", "fact_content_query_90d")
df = dataset['train'].to_pandas()
print(df.shape)
print(df.columns.tolist())
df.head()

(2414248, 21)
['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']


,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,...,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,...,11,0,10.818182,NaN,10.818182,1466,14,32,0.043656,0.725102
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,34,7,2026-04-02,2026-06-30,13,0,0,...,1,0,1.769231,NaN,11.000000,1466,14,32,0.043656,0.725102
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_9f0c36a6ae2a6a99,16,2,2026-04-02,2026-06-30,16,0,11,...,5,0,23.562500,24.272727,22.000000,1466,14,32,0.043656,0.725102
3,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_a032820b5467e996,24,4,2026-04-02,2026-06-30,55,0,1,...,1,0,2.200000,13.000000,0.000000,1466,14,32,0.043656,0.725102
4,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_ba1a2f131961c5da,18,3,2026-04-02,2026-06-30,14,0,0,...,0,0,3.428571,NaN,NaN,1466,14,32,0.043656,0.725102


In [24]:
for col in df.columns:
    print(col)

client_hash_id
content_hash_id
query_hash_id
query_char_count
query_token_count
window_start
window_end
impressions_90d
clicks_90d
impressions_last30
clicks_last30
impressions_prev30
clicks_prev30
avg_position_90d
avg_position_last30
avg_position_prev30
content_total_impressions_90d
content_visible_query_count
rare_query_count
rare_impressions_share
anonymized_impressions_share


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [25]:
fields = {
    "feature": [
        "query_char_count", "query_token_count",
        "impressions_last30", "clicks_last30",
        "impressions_prev30", "clicks_prev30",
        "avg_position_last30", "avg_position_prev30",
        "content_total_impressions_90d", "content_visible_query_count",
        "rare_query_count", "rare_impressions_share", "anonymized_impressions_share"
    ],
    "label": ["clicks_90d"],
    "context": ["client_hash_id", "content_hash_id", "query_hash_id", "window_start", "window_end"],
    "excluded": {
        "impressions_90d": "same 90d window as the label -> leaks the outcome",
        "avg_position_90d": "same 90d window as the label -> leaks the outcome"
    }
}
for bucket, items in fields.items():
    print(bucket.upper(), ":", items)

FEATURE : ['query_char_count', 'query_token_count', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']
LABEL : ['clicks_90d']
CONTEXT : ['client_hash_id', 'content_hash_id', 'query_hash_id', 'window_start', 'window_end']
EXCLUDED : {'impressions_90d': 'same 90d window as the label -> leaks the outcome', 'avg_position_90d': 'same 90d window as the label -> leaks the outcome'}


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [26]:
grain_check = df.groupby(['client_hash_id','content_hash_id','query_hash_id']).size()
print("Max rows per (client, content, query):", grain_check.max())
print("Duplicated combos:", (grain_check > 1).sum())

Max rows per (client, content, query): 1
Duplicated combos: 0


In [27]:
import pandas as pd

df['window_start'] = pd.to_datetime(df['window_start'])
df['window_end'] = pd.to_datetime(df['window_end'])

print("Total rows:", len(df))
print("Window span (days):", (df['window_end'] - df['window_start']).dt.days.max())
print("Earliest window_start:", df['window_start'].min())
print("Latest window_end:", df['window_end'].max())

Total rows: 2414248
Window span (days): 89
Earliest window_start: 2026-04-02 00:00:00
Latest window_end: 2026-06-30 00:00:00


In [28]:
is_true = df['impressions_last30'].notna() & (df['impressions_last30'] > 0)
print("Rows with valid last30 impressions:", is_true.sum(), "/", len(df))
print("Survival rate:", round(is_true.mean()*100, 2), "%")

Rows with valid last30 impressions: 1883490 / 2414248
Survival rate: 78.02 %


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can never tell us: (1) individual user identity or the true query text — small queries below a privacy threshold are folded into rare_query_count / anonymized_impressions_share, hiding long-tail intent; (2) the causal reason behind position or click changes; (3) window overlaps exist — last30 and prev30 sit inside the same 90d window as clicks_90d, so features are not fully independent of the label period; (4) history before 2026-04-02 is not visible, so long-term trends can't be checked.

In [29]:
overlap_check = df[['impressions_last30','impressions_prev30','impressions_90d']].isnull().sum()
print("Missing values per window column:\n", overlap_check)

print("\nRows where last30 + prev30 > 90d total (possible overlap issue):")
overlap_issue = (df['impressions_last30'] + df['impressions_prev30']) > df['impressions_90d']
print(overlap_issue.sum(), "/", len(df))

Missing values per window column:
 impressions_last30    0
impressions_prev30    0
impressions_90d       0
dtype: int64

Rows where last30 + prev30 > 90d total (possible overlap issue):
0 / 2414248


## Feature Frame (5 features max)

1. avg_position_last30 — knowable at decision moment because it only uses the prior 30-day window, before clicks_90d is observed
2. impressions_last30 — knowable at decision moment because it reflects visibility already logged before the current label window
3. query_char_count — knowable at decision moment because it's a static property of the query text itself
4. content_visible_query_count — knowable at decision moment because it's a content-level count available before ranking
5. rare_impressions_share — knowable at decision moment because it reflects historical visibility patterns, not the outcome being predicted

In [30]:
feature_frame = df[[
    'client_hash_id', 'content_hash_id', 'query_hash_id',
    'avg_position_last30', 'impressions_last30',
    'query_char_count', 'content_visible_query_count',
    'rare_impressions_share', 'clicks_90d'
]].copy()

print(feature_frame.shape)
feature_frame.head()

(2414248, 9)


,client_hash_id,content_hash_id,query_hash_id,avg_position_last30,impressions_last30,query_char_count,content_visible_query_count,rare_impressions_share,clicks_90d
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,NaN,0,17,14,0.043656,0
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,NaN,0,34,14,0.043656,0
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_9f0c36a6ae2a6a99,24.272727,11,16,14,0.043656,0
3,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_a032820b5467e996,13.000000,1,24,14,0.043656,0
4,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_ba1a2f131961c5da,NaN,0,18,14,0.043656,0


## The Trap: Deliberate Leakage

Adding a label-derived column (avg_position_90d, from the SAME window as clicks_90d) artificially inflates a quick "score" — this is the leakage trap from notebook 02.

In [31]:
import numpy as np

# LEAKY version: uses avg_position_90d which overlaps the label window
feature_frame['leaky_quick_score'] = (
    (feature_frame['clicks_90d'] > 0).astype(int) *
    (100 - df['avg_position_90d'].fillna(100))
)
leaky_corr = feature_frame['leaky_quick_score'].corr(feature_frame['clicks_90d'])
print("LEAKY correlation with label:", round(leaky_corr, 4))

# Remove the leaky column -- honest version
feature_frame = feature_frame.drop(columns=['leaky_quick_score'])
honest_corr = feature_frame['avg_position_last30'].corr(feature_frame['clicks_90d'])
print("HONEST correlation with label:", round(honest_corr, 4))

LEAKY correlation with label: 0.3269
HONEST correlation with label: -0.0669


Result: leaky correlation (0.33) is notably stronger than the honest correlation (-0.07), showing how a label-window feature creates a misleadingly optimistic signal.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.